# BSDT Sonar — H7 FAST: Schedule Comparison

**Trimmed for speed:** 4 key schedules × n=750,1000 only × 50 instances × 1000 particles.  
~45 min total vs 6+ hours for the full version.  
Answers the core question: does power10/power20 beat cosine?

Author: Odeyemi Olusegun Israel, Independent Researcher, Derby UK

In [ ]:
import torch
import numpy as np
import time
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ════════════════════════════════════════════════════════════
# ENGINE
# ════════════════════════════════════════════════════════════

class BSDTSonarEngine:
    def __init__(self, n, num_instances=50, num_particles=1000,
                 alpha=3.0, mu_scale=0.1, device=device):
        self.n = n; self.ni = num_instances; self.np_ = num_particles
        self.alpha = alpha; self.mu_scale = mu_scale; self.device = device
        self.m = int(alpha * n)

    def generate_instances(self):
        ni, m, n = self.ni, self.m, self.n
        cv = torch.zeros(ni, m, 3, dtype=torch.long, device=self.device)
        cs = torch.zeros(ni, m, 3, dtype=torch.float, device=self.device)
        for inst in range(ni):
            for c in range(m):
                perm = torch.randperm(n, device=self.device)[:3]
                cv[inst, c] = perm
                cs[inst, c] = torch.randint(0, 2, (3,), device=self.device).float() * 2 - 1
        return cv, cs

    def compute_mu(self, clause_vars):
        degrees = torch.zeros(self.ni, self.n, device=self.device)
        for pos in range(3):
            degrees.scatter_add_(1, clause_vars[:, :, pos],
                                  torch.ones(self.ni, self.m, device=self.device))
        lam_max = 0.25 * degrees.max(dim=1).values
        return (self.mu_scale * lam_max).clamp(min=0.01), lam_max

    def energy_and_grad(self, s, cv, cs, mu):
        ni, np_, m, n = self.ni, s.shape[1], self.m, self.n
        cv4  = cv.unsqueeze(1).expand(ni, np_, m, 3)
        s_at = torch.gather(s.unsqueeze(2).expand(ni, np_, m, n), 3, cv4)
        cs4  = cs.unsqueeze(1).expand(ni, np_, m, 3)
        lit  = (1.0 - cs4 * s_at) / 2.0
        E_c  = (lit[...,0] * lit[...,1] * lit[...,2]).sum(dim=2)
        mu3  = mu.view(ni, 1, 1)
        g    = torch.zeros(ni, np_, n, device=self.device)
        l0, l1, l2 = lit[...,0], lit[...,1], lit[...,2]
        for pos, dl in enumerate([
            (-cs4[...,0]/2)*l1*l2,
            l0*(-cs4[...,1]/2)*l2,
            l0*l1*(-cs4[...,2]/2)
        ]):
            g.scatter_add_(2, cv[:,: ,pos].unsqueeze(1).expand(ni, np_, m), dl)
        g = g + mu3 * (-4.0 * s * (1.0 - s**2))
        return E_c + (mu3*(1.0-s**2)**2).sum(dim=2), E_c, g

    def gradient_flow(self, s, cv, cs, mu, steps, dt=0.05,
                      beta=0.90, mu_override=None):
        ni, np_, n = self.ni, s.shape[1], self.n
        v = torch.zeros_like(s)
        plat = torch.zeros(ni, np_, device=self.device)
        best_E = torch.full((ni, np_), float('inf'), device=self.device)
        for step in range(steps):
            mu_eff = mu_override(step, steps, mu) if mu_override else mu
            _, E_c, g = self.energy_and_grad(s, cv, cs, mu_eff)
            imp = E_c < best_E
            best_E = torch.where(imp, E_c, best_E)
            plat = torch.where(imp, torch.zeros_like(plat), plat + 1)
            pm = plat >= 50
            decay = 1.0 / (1.0 + 0.002 * step)
            dt_e  = dt * decay / (1.0 + 0.05 * g.norm(dim=2, keepdim=True).clamp(1e-10))
            dt_e  = dt_e * torch.where(pm.unsqueeze(2),
                                        torch.full_like(dt_e, 2.0),
                                        torch.ones_like(dt_e))
            gamma = (E_c.clamp(0) / (E_c.clamp(0) + 1.0)).unsqueeze(2)
            v = beta * v - dt_e * (1.0 + gamma) * g
            ns = torch.where(pm.unsqueeze(2),
                             torch.full_like(v, 0.03 * decay * 4),
                             torch.full_like(v, 0.03 * decay))
            s = torch.clamp(s + v + torch.randn_like(s) * ns, -1.0, 1.0)
        return s, best_E

    def solve_rate(self, s, cv, cs, mu):
        sr = torch.sign(s + 1e-10)
        _, E, _ = self.energy_and_grad(sr, cv, cs, mu)
        best = E.min(dim=1).values
        rate = (best < 0.5).float().mean().item()
        se   = np.sqrt(rate * (1-rate) / self.ni)
        return rate, se, best.mean().item()

print('Engine loaded.')

In [ ]:
# ════════════════════════════════════════════════════════════
# H7 FAST — 4 KEY SCHEDULES ONLY
#
# Skipped:  n=500 (always 100%, no information)
#           10 extra schedules (tested, will be dominated)
# Reduced:  instances 100→50,  particles 2000→1000
# Est time: ~20 min (n=750) + ~45 min (n=1000) = ~65 min total
# ════════════════════════════════════════════════════════════

# ── Schedules: cosine baseline + 3 best candidates ──────────

def sched_cosine(t):      return (1.0 - np.cos(np.pi * t)) / 2.0
def sched_power10(t):     return t ** 10
def sched_power20(t):     return t ** 20
def sched_delay70(t):
    if t < 0.7: return 0.0
    t2 = (t - 0.7) / 0.3
    return (1.0 - np.cos(np.pi * t2)) / 2.0

SCHEDULES = {
    'cosine':    sched_cosine,    # baseline (H1 proven)
    'power10':   sched_power10,   # 93% broadcast — theory best
    'power20':   sched_power20,   # 97% broadcast — most aggressive
    'delay70':   sched_delay70,   # 85% broadcast — flat first 70%
}

# Broadcast fractions
t_arr = np.linspace(0, 1, 1000)
print('Schedule  |  μ/2  |  μ/4  |  μ/10')
print('-' * 40)
for name, fn in SCHEDULES.items():
    v = np.array([fn(t) for t in t_arr])
    print(f'{name:>10}: {(v<0.50).mean():.0%}    {(v<0.25).mean():.0%}    {(v<0.10).mean():.0%}')


def run_fast(n, num_instances=50, num_particles=1000):
    steps = min(int(500 * np.sqrt(n)), 20000)
    engine = BSDTSonarEngine(n=n, num_instances=num_instances,
                              num_particles=num_particles, device=device)
    cv, cs = engine.generate_instances()
    mu, _  = engine.compute_mu(cv)

    print(f'\n{"="*55}')
    print(f'n={n}  m={int(3*n)}  steps={steps}  '
          f'instances={num_instances}  particles={num_particles}')
    print(f'{"="*55}')

    results = {}
    for name, fn in SCHEDULES.items():
        if device.type == 'cuda': torch.cuda.empty_cache()
        def mu_ov(step, total, mu_b, _fn=fn):
            return mu_b * _fn(step / max(total-1, 1))

        s0 = torch.clamp(torch.randn(num_instances, num_particles, n,
                                      device=device) * 0.3, -0.9, 0.9)
        t0 = time.time()
        sf, _ = engine.gradient_flow(s0, cv, cs, mu, steps, mu_override=mu_ov)
        r, se, viol = engine.solve_rate(sf, cv, cs, mu)
        elapsed = time.time() - t0

        results[name] = {'rate': r, 'se': se, 'viol': viol, 'time': elapsed}
        star = '★' if r >= 0.99 else '◆' if r >= 0.95 else ' '
        print(f'  {star} {name:>10}: {r:6.1%} ± {se:.1%}  viol={viol:.2f}  ({elapsed:.0f}s)')

    return results


# ── RUN ─────────────────────────────────────────────────────
torch.manual_seed(42); np.random.seed(42)

print('\nH7 FAST — Testing 4 schedules at n=750 and n=1000')
print('Skipping n=500 (always 100%)\n')

results = {}
for n in [750, 1000]:
    results[n] = run_fast(n)


# ── SUMMARY ─────────────────────────────────────────────────
H1_cosine = {750: 0.93, 1000: 0.55}   # from H1-EXTENDED

print('\n' + '=' * 55)
print('H7 FAST RESULTS')
print('=' * 55)
print(f'  {"Schedule":>10} | {"n=750":>7} | {"n=1000":>8} | {"verdict"}')
print('  ' + '-' * 50)
for name in SCHEDULES:
    r750  = results[750][name]['rate']
    r1000 = results[1000][name]['rate']
    base750  = H1_cosine[750]
    base1000 = H1_cosine[1000]
    up750  = r750  - results[750]['cosine']['rate']
    up1000 = r1000 - results[1000]['cosine']['rate']
    verdict = '★ WINNER' if (up750 > 0.05 or up1000 > 0.05) else \
              '◆ better' if (up750 > 0.01 or up1000 > 0.01) else '= same'
    print(f'  {name:>10} | {r750:>6.1%} | {r1000:>7.1%} | {verdict}  '
          f'(Δ750={up750:+.1%} Δ1000={up1000:+.1%})')

print()
best_n1000 = max(results[1000], key=lambda k: results[1000][k]['rate'])
best_rate  = results[1000][best_n1000]['rate']
cosine_rate = results[1000]['cosine']['rate']
print(f'  Best at n=1000: {best_n1000}  ({best_rate:.1%})')
print(f'  Cosine baseline: {cosine_rate:.1%}')
print(f'  Net uplift: {best_rate - cosine_rate:+.1%}')
if best_rate - cosine_rate > 0.05:
    print(f'  ★ CONFIRMED: {best_n1000} schedule is significantly better than cosine')
elif best_rate - cosine_rate > 0.01:
    print(f'  ◆ MODEST: {best_n1000} shows small improvement')
else:
    print(f'  = NEGLIGIBLE: cosine is near-optimal — schedule shape not the bottleneck')
    print(f'  → The real bottleneck is wrong-corner convergence → proceed to H8a Nullspace')


# ── CHART ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: schedule curves
ax = axes[0]
colors = ['blue', 'orange', 'red', 'green']
t_plot = np.linspace(0, 1, 500)
for (name, fn), col in zip(SCHEDULES.items(), colors):
    y = np.array([fn(t) for t in t_plot])
    v = np.array([fn(t) for t in t_arr])
    ax.plot(t_plot, y, color=col, lw=2,
            label=f'{name} ({(v<0.1).mean():.0%} below μ/10)')
ax.axhline(0.1, color='k', ls=':', alpha=0.4, label='μ/10')
ax.axhline(0.5, color='k', ls='--', alpha=0.4, label='μ/2')
ax.set_xlabel('t'); ax.set_ylabel('μ(t)/μ_target')
ax.set_title('Schedule Shapes')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Right: solve rates
ax2 = axes[1]
ns_plot = [750, 1000]
for (name, _), col in zip(SCHEDULES.items(), colors):
    rates = [results[n][name]['rate']*100 for n in ns_plot]
    ses   = [results[n][name]['se']*100 for n in ns_plot]
    ax2.errorbar(ns_plot, rates, yerr=ses, marker='o', color=col,
                 lw=2.5, ms=9, capsize=4, label=name)
ax2.set_xlabel('n'); ax2.set_ylabel('Solve rate (%)')
ax2.set_title('H7 Fast: Solve Rates at Frontier')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
ax2.set_xticks([750, 1000])
ax2.set_ylim(40, 105)

plt.tight_layout()
plt.savefig('h7_fast_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h7_fast_results.png')